# `models/burstormer/`：Burstormer Teacher 模組手冊

## 1. 角色與檔案

本模組包裝預訓練 Burstormer 為 **frozen Teacher**。Training 時 Teacher 只 forward 產生 denoised target，不計算參數梯度、不由 optimizer 更新。

| 檔案／路徑 | 用途 |
|---|---|
| `Network.py` | Burstormer model architecture |
| `burstormer_teacher.py` | checkpoint loader、eval/freeze/no-grad wrapper、factory |
| `__init__.py` | 方便外部匯入 `create_teacher` |
| 專案根目錄 `weights/Burstormer.ckpt` | 預訓練權重，不在此子資料夾 |

`Color_denoising_testing.py` 是上游 benchmark/testing entry，不是本專案 KD trainer 的依賴。本專案使用 `Network.py` 和 Teacher adapter；沒有把上游整個 testing dataset、post-processing 或 metrics pipeline 接進來。

## 2. Teacher 輸入／輸出契約

Burstormer 接受的輸入為：

```text
[B, K, C, H, W] = [1, 8, 6, H, W]
C = 3 RGB channels + 3 per-pixel sigma-estimate channels
```

輸出為 `[1, 3, H, W]`。目前本地 `Network.py` 的 forward 會選取 burst batch 中第一個樣本，因此 `B` 必須為 1；正式 `train_kd.py` DataLoader 對應設定 `batch_size=1`。

Sigma estimate 是每個 burst frame、每個 RGB channel 對應的額外輸入通道，不是實際噪聲 tensor。它要描述此次 noisy burst 的預期標準差。正式 KD Dataset 對每筆合成 noise 使用同次抽樣的 `sigma_read`／`sigma_shot` 計算 sigma map，並與 noisy RGB burst 串接後送 Teacher。

`synthetic_smoke_dataset.py` 是例外：它使用固定 `0.01` placeholder sigma 及隨機 target，只用來測輸入 shape、forward、loss/backward。不要將 placeholder 當成正式 Teacher noise estimate。

## 3. Checkpoint 載入和執行方式

預設 checkpoint path：

```text
weights/Burstormer.ckpt
```

由 `burstormer_teacher.py` 的 `create_teacher(device, checkpoint_path=None)` 載入。未給路徑時，會以 adapter 檔案位置推導專案根目錄並使用預設權重。正式 trainer 透過這個 factory 初始化 Teacher，之後將其設為 `eval()` 並 freeze parameters。

Smoke test 範例（由專案根目錄執行）：

```powershell
python run_kd_smoke.py `
  --teacher-factory models.burstormer.burstormer_teacher:create_teacher `
  --student-factory models.nafnet.nafnet_student:create_student `
  --device cpu
```

正式 train entry 使用：

```powershell
python train_kd.py --device cuda
```

上式要求 GPU 環境已有 CUDA-enabled PyTorch/torchvision，而且 `torch.cuda.is_available()` 回傳 True。作者 CPU-only 筆電應改用 `--device cpu` 執行短流程確認；正式訓練交由組員的 GPU 機器。

## 4. 已驗證範圍

專案中的 `weights/Burstormer.ckpt` 已在 CPU 上成功載入並完成 Teacher forward：

- 使用 synthetic placeholder input 完成 KD smoke step。
- 使用一筆 Open Images 衍生的實際 KD Dataset sample 與 NAFNet 完成更新。
- 在 9 筆 usable images 上完成 1 epoch CPU training。

Lightning 載入時提示 checkpoint 是舊版本並在載入期間 migration；同時有舊 callback state 的 warning。模型仍完成載入、forward 和訓練步驟；以上測試沒有把原始 checkpoint 改寫。這些是 runtime compatibility 的證據，不是 Teacher restoration quality 或 benchmark parity 的證據。

## 5. 不要從 Smoke Test 推論的事項

Smoke test 使用隨機圖片和固定 sigma placeholder，不能證明：

- `sigma_estimate` 和此次 noisy image 的 noise level 相符。
- Open Images 合成 noise 分佈等同 Teacher 預訓練分佈。
- gamma、linearization、white-level 與強度範圍正確。
- Teacher 輸出達到論文指標，或 Teacher 優於 Student。

正式 Dataset 的 sigma map 與 generator 的採樣參數在程式上相符；但仍需與組員核對 Teacher training/testing preprocessing 的值域與參數單位。論文列出的四組測試噪聲設定不應直接當成目前訓練生成器的唯一設定。

## 6. 上游檔案邊界

目前 KD training 不需複製整個 Burstormer repository。`Network.py` 和已配置的 checkpoint 是模型執行所需部分；上游 `Color_denoising_testing.py`、benchmark Dataset、metric 和 post-process 未接入本專案。若後續要對齊官方 benchmark，應先比對官方 test loader 的輸入轉換與本專案 Dataset，再決定要移植哪些 evaluation behavior，而不是直接把 testing script 當作 KD training script。

## 7. 正式實驗前尚待確認

1. Burstormer 預訓練資料與目前 synthetic noise generator 的 read/shot parameter、value range、gamma/white-level 是否相容。
2. 輸出是否需要與官方 evaluator 相同的 clamp、反轉換或 post-process。
3. 正式 train/validation/test split 和 paper 的四組測試 noise variants 如何採用。
4. test metrics、sample count、20K 定義和儲存報告格式。

目前 adapter 可用且 batch/shape 契約清楚；以上實驗協議和評分仍由後續正式驗證決定。

## 8. Adapter 的訓練模式與 freeze 行為

`BurstormerTeacher` 建構時呼叫 Lightning `load_from_checkpoint`，帶入 color mode，將模型移到指定 device，切換 evaluation mode 並把 parameters 設為 `requires_grad=False`。`train()` 也被覆寫，以避免外部 trainer 呼叫 `teacher.train()` 時意外把 Teacher 改回 train mode。Forward 會檢查 batch size 和 channel count，並在 `torch.no_grad()` 之下執行模型。

這些保護使 KD trainer 不需自己管理 Teacher gradient。不要將 Teacher parameters 放入 Student optimizer。Teacher checkpoint 應維持只讀；訓練輸出只保存 Student checkpoint。

## 9. Factory 用法與 path 規則

正式 trainer 使用 `models.burstormer.burstormer_teacher:create_teacher`，並以 keyword arguments 傳遞 `device` 和 checkpoint 路徑。未提供 checkpoint 時，adapter 依自身檔案推導專案根目錄，預設讀取 `weights/Burstormer.ckpt`。因此從專案根目錄執行最容易理解，但預設 checkpoint path 不依賴目前工作目錄。

若要手動測 factory，可在專案根目錄執行 Python 並呼叫：

```python
from models.burstormer.burstormer_teacher import create_teacher

teacher = create_teacher(device="cpu")
teacher.eval()
```

這只會建立 Teacher；實際 inference 還必須提供 layout 正確、數值合理且與模型預期相符的 burst tensor。不要使用未經確認的 arbitrary random input 得到 output，就宣稱 Teacher 品質已驗證。

## 10. 舊 Checkpoint warning 的處理原則

目前載入 Burstormer checkpoint 時，Lightning 提示 checkpoint 由舊版本升級到目前版本，並警告多個 `ModelCheckpoint` callback state key 在升級後可能碰撞。已觀察到同一 checkpoint 在 CPU smoke、實際資料 KD step 和 1 epoch training 都能載入並完成 forward/update；使用者提供的 log 亦顯示 1 epoch 正常結束。

Warning 表示 checkpoint metadata 版本轉換或訓練 callback state 恢復可能有相容性差異。此專案只做 inference/frozen Teacher，不會恢復原 checkpoint 的 optimizer 或 callbacks，所以目前警告沒有阻止 KD training。除非要維護一份已確認可用備份，否則不要直接執行 Lightning 提示的永久 upgrade 命令改寫 Teacher checkpoint。

## 11. Teacher 輸出和 loss 的對齊

Teacher output 是對 burst reference frame 的 denoising 預測，shape `[1, 3, H, W]`。KD loss 比較 Student output 和 Teacher output；GT loss 比較 Student output 和 clean reference。三者要對同一 reference frame、相同 crop 與相同 H/W。Teacher 輸出中若有 clamp 或 model-internal range handling，會影響 KD target 的尺度；正式 evaluation 若採額外 inverse color transform 或 display post-process，須獨立定義，不能任意只套在其中一側。

本專案目前已確認 shape contract 和程式能跑，尚未對 Teacher output 的 numeric distribution、視覺品質或 paper benchmark post-process 做正式資料評估。後續可以先固定一筆 validation sample，記錄 noisy/clean/teacher output 的 min/max/mean、有限值、sigma map 範圍，再讓組員核對官方 preprocessing。

## 12. 檔案交接 checklist

- 保留 `Network.py`、`burstormer_teacher.py`、`__init__.py`。
- 保留根目錄 `weights/Burstormer.ckpt`，或確保收件者能取得同一版本權重。
- 不需複製上游完整 test dataset、`Color_denoising_testing.py` 或 evaluation utilities 才能啟動本專案 KD trainer。
- `requirements.txt` 需安裝 PyTorch、torchvision、pytorch-lightning、einops 等依賴；GPU 機器的 PyTorch/CUDA build 由組員依環境選擇。
- 先用少量 clean images、`--epochs 1` 做 integration check，再安排正式大規模 run。

Teacher 能載入不代表其 checkpoint 在研究上適用於目前 noise generator；data-domain compatibility 和 benchmark scoring 仍需核查。
